# Step 1: Data Loading & Joining

Goal: combine the four raw CSVs (`provider_lables`, `beneficiary_raw`, `inpatient_raw`, `outpatient_raw`) into a single **claim-level** table — one row per claim, carrying that claim's beneficiary details and its provider's fraud label.

See `docs/DATA_DICTIONARY.md` for what every column in the raw files means.

## Why claim-level, not provider-level, at this stage?

The target label lives at the **Provider** grain, but Inpatient/Outpatient claims live at the **Claim** grain, and Beneficiary data lives at the **Beneficiary** grain. Aggregating straight up to one row per provider here would throw away information before we've even looked at it. Instead:

```
Train.csv (Provider, PotentialFraud)
     |  Provider ID
     v
Inpatient/Outpatient claims (Provider, BeneID, ClaimID, ...)
     |  BeneID
     v
Beneficiarydata (BeneID, DOB, ChronicCond_*, ...)
```

We join everything down to claim-level first, explore and clean at that level, and only aggregate up to provider-level explicitly in Step 4 (feature engineering) — where the aggregation choices can be made deliberately rather than baked in too early.

In [1]:
import sys
sys.path.append('..')

from src.data_loader import build_claim_level_dataset

df = build_claim_level_dataset()
df.shape

(558211, 56)

**What this cell does:** calls the full loading pipeline from `src/data_loader.py`, which:
1. Loads the 4 raw CSVs
2. Stacks Inpatient + Outpatient claims into one table (adding a `ClaimType` column so we don't lose track of which is which)
3. Merges in beneficiary details on `BeneID`
4. Merges in the provider fraud label on `Provider`

Both merges use `validate="many_to_one"` as a safety check — this raises an error immediately if, say, a `BeneID` unexpectedly appears more than once in the beneficiary table, rather than silently duplicating claim rows and inflating the row count.

**Expected output:** `(558211, 56)` — 558,211 claims across 56 columns.

In [2]:
df.head()

,BeneID,ClaimID,ClaimStartDt,ClaimEndDt,Provider,InscClaimAmtReimbursed,AttendingPhysician,OperatingPhysician,OtherPhysician,AdmissionDt,...,ChronicCond_Diabetes,ChronicCond_IschemicHeart,ChronicCond_Osteoporasis,ChronicCond_rheumatoidarthritis,ChronicCond_stroke,IPAnnualReimbursementAmt,IPAnnualDeductibleAmt,OPAnnualReimbursementAmt,OPAnnualDeductibleAmt,PotentialFraud
0,BENE11001,CLM46614,2009-04-12,2009-04-18,PRV55912,26000,PHY390922,NaN,NaN,2009-04-12,...,1,1,2,1,1,36000,3204,60,70,Yes
1,BENE11001,CLM66048,2009-08-31,2009-09-02,PRV55907,5000,PHY318495,PHY318495,NaN,2009-08-31,...,1,1,2,1,1,36000,3204,60,70,No
2,BENE11001,CLM68358,2009-09-17,2009-09-20,PRV56046,5000,PHY372395,NaN,PHY324689,2009-09-17,...,1,1,2,1,1,36000,3204,60,70,No
3,BENE11011,CLM38412,2009-02-14,2009-02-22,PRV52405,5000,PHY369659,PHY392961,PHY349768,2009-02-14,...,1,2,2,1,1,5000,1068,250,320,No
4,BENE11014,CLM63689,2009-08-13,2009-08-30,PRV56614,10000,PHY379376,PHY398258,NaN,2009-08-13,...,2,1,2,2,2,21260,2136,120,100,No


In [3]:
print(f"Unique providers: {df['Provider'].nunique()}")
print(f"Unique beneficiaries: {df['BeneID'].nunique()}")

Unique providers: 5410
Unique beneficiaries: 138556


**What this checks:** confirms the join scale matches the known dataset size — **5,410 unique providers** and **138,556 unique beneficiaries**. If these numbers were off (e.g. far more providers than 5,410), that would signal a broken join, since the `provider_labels` file only contains 5,410 provider rows to begin with.

In [4]:
df['PotentialFraud'].value_counts(normalize=True)

PotentialFraud
No     0.618789
Yes    0.381211
Name: proportion, dtype: float64

**What this shows:** the fraud label distribution *at claim level* — roughly 38% of claims belong to a fraudulent provider, 62% do not.

**Important nuance:** this is noticeably higher than the ~9% *provider-level* fraud rate quoted for this dataset. That's not a bug — it means fraudulent providers generate a disproportionately large share of claims relative to their share of providers (only ~9% of providers are bad actors, but they account for over a third of all claims). This is worth keeping as a mini-insight for the EDA/report section: claim volume per provider may itself be a useful fraud signal, which Step 3 and Step 4 build on.

In [5]:
df['ClaimType'].value_counts()

ClaimType
Outpatient    517737
Inpatient      40474
Name: count, dtype: int64

**What this shows:** how many claims came from the Inpatient table vs the Outpatient table after stacking — a quick sanity check that the `combine_claims()` step worked as expected (both claim types should be present, with Outpatient typically far more numerous than Inpatient, since outpatient visits are more frequent than hospital admissions).

## Summary of Step 1

- Joined 4 raw tables into a single claim-level dataset: **558,211 rows × 56 columns**
- Verified join integrity via `validate="many_to_one"` and provider/beneficiary counts matching expectations
- Noted an important distinction: **claim-level fraud rate (~38%) ≠ provider-level fraud rate (~9%)** — fraudulent providers generate disproportionately more claims
- Output feeds directly into Step 2 (Cleaning) — no rows or columns are dropped at this stage; that's deferred until we know more about the data